In [ ]:
# Config: hyperparameters, seeds, determinism flags, output paths

import numpy as np, pandas as pd
import os, glob, json, hashlib, warnings, time as timer
from collections import defaultdict
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.stats import ttest_rel, wilcoxon
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.float32
OUTD     = "/kaggle/working"
SPLITDIR = f"{OUTD}/splits"
RESDIR   = f"{OUTD}/results"
for d in (SPLITDIR, RESDIR): os.makedirs(d, exist_ok=True)

RUNS          = 5
TOPICS_TO_RUN = ['Algebra','Fractions','Geometry','Linear','Arithmetic','Statistics']
MAX_LEN       = 200
MIN_INTERACT  = 3
TEST_SIZE     = 0.2
N_TIERS, REF_TIER = 10, 4
R, L          = 2, 2
PC_EPOCHS, PC_LR, PC_SHRINK = 60, 0.08, 1e-3
PC_CLIP       = 5.0
DKT_HIDDEN, DKT_EPOCHS, DKT_LR, DKT_BATCH, DKT_DROPOUT = 100, 30, 1e-3, 64, 0.2
DKT_VAL_FRAC  = 0.1
USE_LEARNABLE_LEAF = True

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
print("torch", torch.__version__, "| device:", device)

In [ ]:
# Load ASSISTments 2012-13, filter to original problems and valid outcomes

DATA_FILE = "2012-2013-data-with-predictions-4-final.csv"
if not os.path.exists(DATA_FILE):
    os.system("kaggle datasets download -d nicolaswattiez/skillbuilder-data-2009-2010 -q")
    os.system("unzip -o skillbuilder-data-2009-2010.zip -d . > /dev/null 2>&1")

df = pd.read_csv(DATA_FILE, encoding="ISO-8859-1", low_memory=False)
df = df[df["original"] == 1].copy()
df = df.dropna(subset=["skill", "correct"])
df["correct"] = df["correct"].astype(int)
df = df[df["correct"].isin([0, 1])]
PTYPE_LIST   = df['problem_type'].dropna().value_counts().index.tolist()
PTYPE_TO_IDX = {p: i for i, p in enumerate(PTYPE_LIST)}
N_PTYPES     = len(PTYPE_LIST)
print(f"Loaded {df.shape[0]:,} rows | {df['user_id'].nunique():,} students | "
      f"{df['skill'].nunique()} skills | {N_PTYPES} problem types | "
      f"correct rate {df['correct'].mean():.4f}")

In [ ]:
# Six CCSS topic subsets; foundational/advanced cluster assignment; per-split dataset builder

TOPIC_PREREQ_SPLIT = {
    'Algebra': {'early': ['Combining Like Terms','Distributive Property','Simplifying Expressions positive exponents','Equation Solving Two or Fewer Steps'],
                'late': ['Factoring Polynomials Standard','Factoring Trinomials','Multiplying non Monomial Polynomials','Polynomial Factors','Equation Solving More Than Two Steps','Quadratic Formula to Solve Quadratic Equation','Solve Quadratic Equations Using Factoring']},
    'Fractions': {'early': ['Equivalent Fractions','Addition and Subtraction Fractions','Multiplication Fractions','Fraction Of'],
                  'late': ['Finding Percents','Percent Of','Conversion of Fraction Decimals Percents','Unit Rate','Proportion','Percent Discount','Percent Increase or Decrease']},
    'Geometry': {'early': ['Angles - Obtuse, Acute, and Right','Complementary and Supplementary Angles','Properties and Classification Triangles'],
                 'late': ['Angles on Parallel Lines Cut by a Transversal','Parallel and Perpendicular Lines','Interior Angles Triangle','Interior Angles Figures with More than 3 Sides','Pythagorean Theorem','Calculations with Similar Figures']},
    'Linear': {'early': ['Recognize Linear Pattern','Finding Slope from Graph','Comparing and Identifying Slope/Rate of Change','Finding Slope from Ordered Pairs','Finding Slope From Situation','Finding Slope From Equation'],
               'late': ['Write Linear Equation from Graph','Write Linear Equation from Ordered Pairs','Write Linear Equation from Slope and y-intercept','Finding y-intercept from Linear Equation','Graphing Linear Equations','Solving Systems of Linear Equations','Solving Systems of Linear Equations by Graphing']},
    'Arithmetic': {'early': ['Addition Whole Numbers','Subtraction Whole Numbers','Multiplication Whole Numbers','Division Whole Numbers','Number Line'],
                   'late': ['Addition and Subtraction Integers','Multiplication and Division Integers','Ordering Integers','Absolute Value','Ordering Real Numbers','Divisibility Rules','Prime Number','Rounding']},
    'Statistics': {'early': ['Mean','Median','Mode','Range'],
                   'late': ['Mean-Median-Mode-Range Differentiation','Histogram as Table or Graph','Box and Whisker','Scatter Plot']},
}

def generate_splits(df, runs=RUNS, test_size=TEST_SIZE, force=False):
    all_users = np.array(sorted(df['user_id'].unique()))
    for run in range(runs):
        path = f"{SPLITDIR}/split_run{run}.csv"
        if os.path.exists(path) and not force: continue
        tr, te = train_test_split(all_users, test_size=test_size, random_state=run)
        pd.concat([pd.DataFrame({'user_id': tr, 'fold': 'train'}),
                   pd.DataFrame({'user_id': te, 'fold': 'test'})]).to_csv(path, index=False)
        print(f"  wrote {path} (train {len(tr):,} / test {len(te):,})")

def load_split(run):
    s = pd.read_csv(f"{SPLITDIR}/split_run{run}.csv")
    return set(s.loc[s.fold=='train','user_id']), set(s.loc[s.fold=='test','user_id'])

generate_splits(df)

def build_dataset_from_split(df, early, late, run, min_interactions=MIN_INTERACT):
    train_ids, test_ids = load_split(run)
    pe = [s for s in early if s in df['skill'].unique()]
    pl = [s for s in late  if s in df['skill'].unique()]
    order = pe + pl; s2id = {s:i for i,s in enumerate(order)}
    sub = df[df['skill'].isin(order)].copy().sort_values(['start_time'])

    tr_raw, te_raw = [], []
    for uid, grp in sub.groupby('user_id'):
        seq = [(s2id[r['skill']], int(r['correct']), PTYPE_TO_IDX.get(r['problem_type'],0),
                r['template_id']) for _, r in grp.iterrows() if r['skill'] in s2id]
        if len(seq) < min_interactions: continue
        if   uid in train_ids: tr_raw.append(seq)
        elif uid in test_ids:  te_raw.append(seq)

    tpl_stats = defaultdict(lambda:[0,0])
    for seq in tr_raw:
        for (k,x,pt,tpl) in seq:
            if pd.notna(tpl): tpl_stats[tpl][0]+=x; tpl_stats[tpl][1]+=1
    tpl_rate = {t:c/n for t,(c,n) in tpl_stats.items() if n>=5}
    if len(tpl_rate) >= N_TIERS:
        qs = np.quantile(list(tpl_rate.values()), np.linspace(0,1,N_TIERS+1))
        def tier_of(tpl):
            r = tpl_rate.get(tpl)
            return REF_TIER if r is None else int(np.clip(np.searchsorted(qs[1:-1], r),0,N_TIERS-1))
    else:
        def tier_of(tpl): return REF_TIER
    tpl_index = {t:i+1 for i,t in enumerate(sorted(tpl_rate.keys()))}
    def tid_of(tpl): return tpl_index.get(tpl, 0)
    def enrich(seqs): return [[(k,x,pt,tier_of(tpl),tid_of(tpl)) for (k,x,pt,tpl) in seq] for seq in seqs]

    return {'n_skills':len(order), 'cluster_of':np.array([0]*len(pe)+[1]*len(pl)),
            'n_templates':len(tpl_index)+1, 'train':enrich(tr_raw), 'test':enrich(te_raw)}

In [ ]:
# ==== CELL 4: PC model ====
# Probabilistic circuit student model and its training.
#   Structure: root sum over R=2 archetypes -> product over the two skill clusters
#   (foundational/advanced) -> per-cluster sum over L=2 proficiency states -> Bernoulli
#   mastery leaf per skill.
#   Learned components: archetype priors, per-archetype cluster weights (random init to
#   break symmetry), per-skill learning rate, forgetting rate, ordered leaf priors, and a
#   per-template Rasch-style difficulty scalar (latent mode).
#   Transition: theta' = theta*(1-f) + (1-theta)*learn.
#   Emission: additive in logit space (per-skill baseline + problem-type offset + difficulty).
#   Training: exact forward algorithm over (time, student) tensors, Adam, with L2 shrinkage
#   on the difficulty scalars; all parameters fit jointly by maximizing marginal log-likelihood.

class GLMParams(torch.nn.Module):
    # CHANGED: __init__ now takes C (num clusters) and R (num archetypes) so cluster
    # weights and root prior can be learned parameters instead of frozen constants.
    def __init__(self, K, n_types, n_tiers, C, R):
        super().__init__()
        self.skill_guess_logit=torch.nn.Parameter(torch.full((K,),-1.4,dtype=dtype))
        self.skill_slip_logit =torch.nn.Parameter(torch.full((K,),-2.0,dtype=dtype))
        self.type_guess_free=torch.nn.Parameter(torch.zeros(n_types-1,dtype=dtype))
        self.type_slip_free =torch.nn.Parameter(torch.zeros(n_types-1,dtype=dtype))
        self.tier_guess_free=torch.nn.Parameter(torch.zeros(n_tiers-1,dtype=dtype))
        self.tier_slip_free =torch.nn.Parameter(torch.zeros(n_tiers-1,dtype=dtype))
        # CHANGED (#3): per-skill learn rate, was a single scalar torch.tensor(-1.7)
        self.learn_raw =torch.nn.Parameter(torch.full((K,),-1.7,dtype=dtype))
        self.forget_raw=torch.nn.Parameter(torch.tensor(-4.0,dtype=dtype))
        # CHANGED (#2): learned root/archetype prior, was fixed [0.5,0.5]
        self.root_prior_logit =torch.nn.Parameter(torch.zeros(R,dtype=dtype))
        # CHANGED (#1): learned cluster-weight-high (R,C), was frozen [[0.75],[0.25]].
        # Random init breaks archetype symmetry ( fix).
        self.cluster_weight_high_logit =torch.nn.Parameter(0.1*torch.randn(R,C,dtype=dtype))
        self.tpl_diff=None
        self.leaf_a=torch.nn.Parameter(torch.full((0,),0.0,dtype=dtype))
        self.leaf_b=torch.nn.Parameter(torch.full((0,),0.0,dtype=dtype))
    def enable_leaf(self,K):
        self.leaf_a=torch.nn.Parameter(torch.full((K,),-1.1,dtype=dtype))
        self.leaf_b=torch.nn.Parameter(torch.full((K,),1.4,dtype=dtype))
    def leaf(self):
        low=torch.sigmoid(self.leaf_a); high=low+(1-low)*torch.sigmoid(self.leaf_b)
        return torch.stack([low,high],dim=1)
    def enable_latent(self,n): self.tpl_diff=torch.nn.Parameter(torch.zeros(n,dtype=dtype))
    def _wr(self,free,ref):
        z=torch.zeros(1,dtype=dtype,device=free.device); return torch.cat([free[:ref],z,free[ref:]])
    def tgo(self): return self._wr(self.type_guess_free,0)
    def tso(self): return self._wr(self.type_slip_free,0)
    def dgo(self): return self._wr(self.tier_guess_free,REF_TIER)
    def dso(self): return self._wr(self.tier_slip_free,REF_TIER)
    def learn(self): return torch.sigmoid(self.learn_raw)          # CHANGED (#3): now returns (K,) vector
    def forget(self): return torch.sigmoid(self.forget_raw)
    def root_prior(self): return torch.softmax(self.root_prior_logit,dim=0)         # CHANGED (#2)
    def cluster_weight_high(self): return torch.sigmoid(self.cluster_weight_high_logit)  # CHANGED (#1), (R,C)


def to_batch(seqs, K):
    B=len(seqs); T=min(MAX_LEN,max(len(s) for s in seqs))
    sk=np.zeros((T,B),np.int64); rs=np.zeros((T,B),np.int64); pt=np.zeros((T,B),np.int64)
    dt=np.full((T,B),REF_TIER,np.int64); tp=np.zeros((T,B),np.int64); mk=np.zeros((T,B),np.float32)
    for b,seq in enumerate(seqs):
        for t,(k,x,p,d,ti) in enumerate(seq[:T]):
            sk[t,b]=k;rs[t,b]=x;pt[t,b]=p;dt[t,b]=d;tp[t,b]=ti;mk[t,b]=1.0
    tt=lambda a,dd: torch.tensor(a,dtype=dd,device=device)
    return (tt(sk,torch.long),tt(rs,torch.long),tt(pt,torch.long),
            tt(dt,torch.long),tt(tp,torch.long),tt(mk,dtype))

# CHANGED: cwh_t and rp_t params removed from signature — now read from `params`.
def run_batch(params, sk,rs,pt,dt,tp,mk, K,C,col_t,leaf_t,
              mode='tiers', no_forget=False):
    B=sk.shape[1]
    rp_now=params.root_prior()                       # CHANGED (#2)
    W=rp_now.unsqueeze(0).repeat(B,1).clone()
    cwh_now=params.cluster_weight_high()             # CHANGED (#1), (R,C)
    Q=torch.zeros(B,C,R,L,dtype=dtype,device=device)
    for c in range(C): Q[:,c,:,1]=cwh_now[:,c]; Q[:,c,:,0]=1-cwh_now[:,c]
    leaf_now = params.leaf() if params.leaf_a.numel()>0 else leaf_t
    theta=leaf_now.unsqueeze(0).unsqueeze(2).repeat(B,1,R,1)
    tgo,tso,dgo,dso=params.tgo(),params.tso(),params.dgo(),params.dso()
    lr_vec=params.learn()                            # CHANGED (#3): (K,) per-skill
    fg=None if no_forget else params.forget()
    total=torch.zeros((),dtype=dtype,device=device)
    for t in range(sk.shape[0]):
        k=sk[t];x=rs[t];p=pt[t];d=dt[t];ti=tp[t];m=mk[t]
        oh=F.one_hot(k,K).to(dtype); coh=F.one_hot(col_t[k],C).to(dtype)
        th=(theta*oh[:,:,None,None]).sum(1)
        lr=lr_vec[k][:,None,None]                    # CHANGED (#3): per-skill learn rate this step
        # EXACT f=0 ablation: drop the (1-f) decay factor entirely
        th = th + (1-th)*lr if no_forget else th*(1-fg) + (1-th)*lr
        if mode=='latent':
            dd=params.tpl_diff[ti]
            g=torch.sigmoid(params.skill_guess_logit[k]+tgo[p]-dd)
            s=torch.sigmoid(params.skill_slip_logit[k]+tso[p]+dd)
        else:
            g=torch.sigmoid(params.skill_guess_logit[k]+tgo[p]+dgo[d])
            s=torch.sigmoid(params.skill_slip_logit[k]+tso[p]+dso[d])
        Qt=(Q*coh[:,:,None,None]).sum(1)
        pc=th*(1-s)[:,None,None]+(1-th)*g[:,None,None]
        lik=torch.where(x[:,None,None]==1,pc,1-pc)
        lik_r=(Qt*lik).sum(2); logl_r=torch.log(torch.clamp(lik_r,min=1e-10))
        nQ=Qt*lik; nQ=nQ/torch.clamp(nQ.sum(2,keepdim=True),min=1e-10)
        p1=torch.where(x==1,1-s,s); p0=torch.where(x==1,g,1-g)
        num=th*p1[:,None,None]; nth=num/torch.clamp(num+(1-th)*p0[:,None,None],min=1e-10)
        mB=m[:,None,None]
        thw=mB*nth+(1-mB)*(theta*oh[:,:,None,None]).sum(1)
        theta=theta*(1-oh)[:,:,None,None]+thw[:,None,:,:]*oh[:,:,None,None]
        Qw=mB*nQ+(1-mB)*Qt
        Q=Q*(1-coh)[:,:,None,None]+Qw[:,None,:,:]*coh[:,:,None,None]
        ulw=torch.log(torch.clamp(W,min=1e-10))+logl_r*m[:,None]
        mx=ulw.max(1,keepdim=True).values
        logZ=(mx.squeeze(1)+torch.log(torch.exp(ulw-mx).sum(1)))*m
        W=torch.exp(ulw-mx); W=W/W.sum(1,keepdim=True)
        total=total+logZ.sum()
    return total


def train_topic(data, seed, mode='tiers', no_forget=False,
                n_epochs=PC_EPOCHS, lr_adam=PC_LR, shrink=PC_SHRINK):
    torch.manual_seed(seed); np.random.seed(seed)
    K=data['n_skills']; cluster_of=data['cluster_of']; C=int(cluster_of.max())+1
    leaf=np.tile(np.array([0.25,0.85],np.float32),(K,1))
    leaf_t=torch.tensor(leaf,dtype=dtype,device=device)
    col_t =torch.tensor(cluster_of,dtype=torch.long,device=device)
    sk,rs,pt,dt,tp,mk=to_batch(data['train'],K)

    # CHANGED: cwh_t / rp_t no longer built — cluster weights and root prior learned in GLMParams
    params=GLMParams(K,N_PTYPES,N_TIERS,C,R)         # CHANGED: pass C, R
    if USE_LEARNABLE_LEAF: params.enable_leaf(K)
    if mode=='latent':     params.enable_latent(data['n_templates'])
    if no_forget:
        params.forget_raw.requires_grad_(False)
    params=params.to(device)
    opt=torch.optim.Adam([p for p in params.parameters() if p.requires_grad], lr=lr_adam)

    best_state, final_nll = None, None
    for ep in range(n_epochs):
        opt.zero_grad()
        # CHANGED: run_batch call no longer passes cwh_t, rp_t
        loss=-run_batch(params,sk,rs,pt,dt,tp,mk,K,C,col_t,leaf_t,
                        mode=mode, no_forget=no_forget)
        if mode=='latent': loss=loss+shrink*(params.tpl_diff**2).sum()
        if not torch.isfinite(loss):
            print(f"      !! non-finite loss at epoch {ep}; reverting to last finite state")
            break
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in params.parameters() if p.requires_grad], PC_CLIP)
        opt.step()
        final_nll=float(loss)
        best_state={k:v.detach().clone() for k,v in params.state_dict().items()}
    if best_state is not None: params.load_state_dict(best_state)

    with torch.no_grad():
        return {'sg':params.skill_guess_logit.cpu().numpy(),'ss':params.skill_slip_logit.cpu().numpy(),
                'tgo':params.tgo().cpu().numpy(),'tso':params.tso().cpu().numpy(),
                'dgo':params.dgo().cpu().numpy(),'dso':params.dso().cpu().numpy(),
                'learn':params.learn().detach().cpu().numpy(),                  # CHANGED (#3): (K,) vector
                'forget':(0.0 if no_forget else float(params.forget())),
                'leaf':(params.leaf().detach().cpu().numpy() if params.leaf_a.numel()>0 else leaf),
                'cwh':params.cluster_weight_high().detach().cpu().numpy(),       # CHANGED (#1): learned (R,C)
                'rp':params.root_prior().detach().cpu().numpy(),                 # CHANGED (#2): learned (R,)
                'tpl_diff':(params.tpl_diff.cpu().numpy() if params.tpl_diff is not None else None),
                'mode':mode,'nll':final_nll,'no_forget':bool(no_forget),
                'n_params':sum(p.numel() for p in params.parameters() if p.requires_grad)}


def train_topic_best(data, seed, restarts=1, **kw):
    best, best_nll, last = None, float('inf'), None
    for r in range(restarts):
        out = train_topic(data, seed=seed*100+r, **kw); last = out
        nll = out.get('nll')
        if nll is not None and np.isfinite(nll) and nll < best_nll:
            best_nll, best = nll, out
    if best is None:
        print(f"  !! all restarts non-finite; using last fit")
        best = last
    return best

In [ ]:
# ==== CELL 5: Metrics and PC evaluation ====
# all_metrics: AUC, expected calibration error (fixed-width and quantile bins), and Brier score.
# evaluate_pc: causal next-step evaluation of the trained circuit. For each student sequence it
#   runs the forward filter one step at a time, and at each step predicts the response *before*
#   seeing the outcome (predict-then-update), so there is no label leakage. Scores steps t>=1
#   (the first step has no history). With pnp['forget']==0 the transition reduces exactly to the
#   no-forgetting form. zero_tiers/latent control whether difficulty comes from tier offsets or
#   the per-template latent scalar.

def sigmoid_np(x): return 1.0/(1.0+np.exp(-x))

def all_metrics(probs, labels, n_bins=10):
    probs=np.asarray(probs,float); labels=np.asarray(labels,float)
    auc = roc_auc_score(labels, probs) if len(np.unique(labels))>1 else 0.5
    edges=np.linspace(0,1,n_bins+1); ece=0.0
    for i in range(n_bins):
        m=(probs>=edges[i])&(probs<edges[i+1])
        if m.sum()>0: ece += m.sum()*abs(labels[m].mean()-probs[m].mean())
    ece/=len(labels)
    qs=np.quantile(probs,np.linspace(0,1,n_bins+1)); qs[0]-=1e-9; qs[-1]+=1e-9
    eceq=0.0
    for i in range(n_bins):
        m=(probs>qs[i])&(probs<=qs[i+1])
        if m.sum()>0: eceq += m.sum()*abs(labels[m].mean()-probs[m].mean())
    eceq/=len(labels)
    return dict(auc=float(auc), ece=float(ece), ece_q=float(eceq),
                brier=float(np.mean((probs-labels)**2)), n=int(len(labels)))

def evaluate_pc(pnp, cluster_of, test, zero_tiers=False):
    K=len(cluster_of); C=int(cluster_of.max())+1
    latent = pnp.get('mode')=='latent' and pnp.get('tpl_diff') is not None and not zero_tiers
    dgo = np.zeros_like(pnp['dgo']) if (zero_tiers or latent) else pnp['dgo']
    dso = np.zeros_like(pnp['dso']) if (zero_tiers or latent) else pnp['dso']
    probs,labels=[],[]
    for seq in test:
        seq=seq[:MAX_LEN]
        W=pnp['rp'].copy(); Q=[np.zeros((R,L)) for _ in range(C)]
        for c in range(C): Q[c][:,1]=pnp['cwh'][:,c]; Q[c][:,0]=1-pnp['cwh'][:,c]
        theta=[np.tile(pnp['leaf'][k],(R,1)) for k in range(K)]
        for t,(k,x,pt,d,ti) in enumerate(seq):
            # CHANGED (#3): per-skill learn rate pnp['learn'][k] instead of scalar pnp['learn']
            th=theta[k]; th=th*(1-pnp['forget'])+(1-th)*pnp['learn'][k]; theta[k]=th
            if latent:
                dd=pnp['tpl_diff'][ti] if ti<len(pnp['tpl_diff']) else 0.0
                g=sigmoid_np(pnp['sg'][k]+pnp['tgo'][pt]-dd); s=sigmoid_np(pnp['ss'][k]+pnp['tso'][pt]+dd)
            else:
                g=sigmoid_np(pnp['sg'][k]+pnp['tgo'][pt]+dgo[d]); s=sigmoid_np(pnp['ss'][k]+pnp['tso'][pt]+dso[d])
            c=cluster_of[k]; p_m=float((W*(Q[c]*th).sum(axis=1)).sum())
            if t>=1:
                probs.append(np.clip(p_m*(1-s)+(1-p_m)*g,1e-6,1-1e-6)); labels.append(x)
            pc=th*(1-s)+(1-th)*g; lik=pc if x==1 else 1-pc
            lik_r=(Q[c]*lik).sum(axis=1); nQ=Q[c]*lik
            Q[c]=nQ/np.clip(nQ.sum(axis=1,keepdims=True),1e-12,None)
            p1=(1-s) if x==1 else s; p0=g if x==1 else (1-g)
            num=th*p1; theta[k]=num/np.clip(num+(1-th)*p0,1e-12,None)
            ulw=np.log(np.clip(W,1e-12,None))+np.log(np.clip(lik_r,1e-12,None))
            W=np.exp(ulw-ulw.max()); W=W/W.sum()
    return np.array(probs), np.array(labels)

In [ ]:
# ==== BKT baseline: per-topic overall AUC + ECE (5 splits), self-contained ====
# Standard Bayesian Knowledge Tracing fit independently per skill by EM (Baum-Welch).
#   Two latent states (not-mastered / mastered) with an absorbing mastery transition;
#   parameters p_init, p_learn, p_slip, p_guess estimated via forward-backward (_e_step)
#   and M-step updates, with Beta priors and slip/guess bounds (p_slip + p_guess < 1) for
#   identifiability. predict_correctness_probabilities gives causal (predict-before-update)
#   probabilities. This cell fits BKT per topic over the 5 splits and reports mean AUC/ECE.

from dataclasses import dataclass
EPS=1e-9
@dataclass
class BKTParams:
    p_init: float; p_learn: float; p_slip: float; p_guess: float
class BKTModel:
    def __init__(self, n_iter=100, tol=1e-5, init_params=None,
                 slip_bounds=(1e-4,0.3), guess_bounds=(1e-4,0.3), max_slip_plus_guess=0.9,
                 slip_prior_mean=0.1, slip_prior_strength=4.0,
                 guess_prior_mean=0.1, guess_prior_strength=4.0):
        self.n_iter=n_iter; self.tol=tol; self.init_params=init_params
        self.slip_bounds=slip_bounds; self.guess_bounds=guess_bounds
        self.max_slip_plus_guess=max_slip_plus_guess
        self.slip_prior_mean=slip_prior_mean; self.slip_prior_strength=slip_prior_strength
        self.guess_prior_mean=guess_prior_mean; self.guess_prior_strength=guess_prior_strength
        self.params_=None
    @staticmethod
    def _pad(sequences):
        n=len(sequences); max_len=max((len(s) for s in sequences),default=0)
        obs=np.zeros((n,max_len),dtype=np.int8); mask=np.zeros((n,max_len),dtype=bool)
        for i,s in enumerate(sequences):
            Ls=len(s); obs[i,:Ls]=s; mask[i,:Ls]=True
        return obs,mask
    @staticmethod
    def _emission_prob(obs, slip, guess):
        p0=np.where(obs==1,guess,1-guess); p1=np.where(obs==1,1-slip,slip)
        return np.stack([p0,p1],axis=-1)
    def _e_step(self, obs, mask, params):
        N,T=obs.shape
        emis=np.clip(self._emission_prob(obs,params.p_slip,params.p_guess),EPS,1.0)
        trans=np.array([[1-params.p_learn,params.p_learn],[0.0,1.0]])
        prior=np.array([1-params.p_init,params.p_init])
        alpha=np.zeros((N,T,2)); c=np.ones((N,T))
        alpha0=prior[None,:]*emis[:,0,:]; c0=np.clip(alpha0.sum(1),EPS,None)
        alpha[:,0,:]=alpha0/c0[:,None]; c[:,0]=np.where(mask[:,0],c0,1.0)
        for t in range(1,T):
            pred=alpha[:,t-1,:]@trans; a_t=pred*emis[:,t,:]; c_t=np.clip(a_t.sum(1),EPS,None)
            ar=a_t/c_t[:,None]; m=mask[:,t]
            alpha[:,t,:]=np.where(m[:,None],ar,alpha[:,t-1,:]); c[:,t]=np.where(m,c_t,1.0)
        beta=np.ones((N,T,2))
        for t in range(T-2,-1,-1):
            nxt=beta[:,t+1,:]*emis[:,t+1,:]
            br=(nxt@trans.T)/np.clip(c[:,t+1],EPS,None)[:,None]; m=mask[:,t+1]
            beta[:,t,:]=np.where(m[:,None],br,beta[:,t+1,:])
        gamma=alpha*beta; gamma=gamma/np.clip(gamma.sum(-1,keepdims=True),EPS,None)
        xi=np.zeros((N,max(T-1,0),2,2))
        for t in range(T-1):
            for i in range(2):
                for j in range(2):
                    xi[:,t,i,j]=(alpha[:,t,i]*trans[i,j]*emis[:,t+1,j]*beta[:,t+1,j]/np.clip(c[:,t+1],EPS,None))
        gamma=gamma*mask[:,:,None]
        if T>1: xi=xi*mask[:,1:][:,:,None,None]
        return gamma,xi,0.0
    def _m_step(self, obs, mask, gamma, xi):
        n=obs.shape[0]
        p_init=gamma[:,0,1].sum()/max(n,1)
        xs=xi.sum((0,1)); from0=xs[0,0]+xs[0,1]; p_learn=xs[0,1]/max(from0,EPS)
        mw=gamma[:,:,1]; nw=gamma[:,:,0]; inc=(obs==0); cor=(obs==1)
        ss=(mw*inc*mask).sum(); st=(mw*mask).sum(); gs=(nw*cor*mask).sum(); gt=(nw*mask).sum()
        p_slip=(ss+self.slip_prior_mean*self.slip_prior_strength)/max(st+self.slip_prior_strength,EPS)
        p_guess=(gs+self.guess_prior_mean*self.guess_prior_strength)/max(gt+self.guess_prior_strength,EPS)
        pc=lambda v: float(np.clip(v,1e-4,1-1e-4))
        p_init=pc(p_init); p_learn=pc(p_learn)
        p_slip=float(np.clip(p_slip,*self.slip_bounds)); p_guess=float(np.clip(p_guess,*self.guess_bounds))
        tot=p_slip+p_guess
        if tot>self.max_slip_plus_guess:
            sc=self.max_slip_plus_guess/tot; p_slip*=sc; p_guess*=sc
        return BKTParams(p_init,p_learn,p_slip,p_guess)
    def fit(self, sequences):
        sequences=[s for s in sequences if len(s)>0]
        if not sequences: raise ValueError("empty")
        obs,mask=self._pad(sequences)
        params=self.init_params or BKTParams(0.3,0.15,0.1,0.25)
        for it in range(self.n_iter):
            gamma,xi,_=self._e_step(obs,mask,params); params=self._m_step(obs,mask,gamma,xi)
        self.params_=params; return self
    def predict_correctness_probabilities(self, sequences):
        obs,mask=self._pad(sequences); N,T=obs.shape; p=self.params_
        trans=np.array([[1-p.p_learn,p.p_learn],[0.0,1.0]]); prior=np.array([1-p.p_init,p.p_init])
        emis=np.clip(self._emission_prob(obs,p.p_slip,p.p_guess),EPS,1.0)
        belief=np.tile(prior,(N,1)); preds=np.zeros((N,T))
        for t in range(T):
            preds[:,t]=belief[:,0]*p.p_guess+belief[:,1]*(1-p.p_slip)
            upd=belief*emis[:,t,:]; upd=upd/np.clip(upd.sum(1,keepdims=True),EPS,None)
            belief=upd@trans
        return [preds[i,:int(mask[i].sum())] for i in range(N)]

def bkt_overall_predictions(data):
    K=data['n_skills']
    tr={k:[] for k in range(K)}; te={k:[] for k in range(K)}
    for seq in data['train']:
        per={}
        for (k,x,pt,d,ti) in seq: per.setdefault(k,[]).append(int(x))
        for k,a in per.items(): tr[k].append(np.array(a,dtype=np.int8))
    for seq in data['test']:
        per={}
        for (k,x,pt,d,ti) in seq: per.setdefault(k,[]).append(int(x))
        for k,a in per.items(): te[k].append(np.array(a,dtype=np.int8))
    yt,yp=[],[]
    for k in range(K):
        if not tr[k] or not te[k]: continue
        try:
            m=BKTModel(n_iter=100).fit([s for s in tr[k] if len(s)>0])
            preds=m.predict_correctness_probabilities(te[k])
        except Exception: continue
        for ta,pa in zip(te[k],preds):
            yt.append(ta.astype(int)); yp.append(pa)
    if not yt: return np.array([]),np.array([])
    return np.concatenate(yp), np.concatenate(yt)   # (preds, labels)

bkt_rows=[]
for name in TOPICS_TO_RUN:
    sp=TOPIC_PREREQ_SPLIT[name]; aucs,eces=[],[]
    for run in range(RUNS):
        data=build_dataset_from_split(df, sp['early'], sp['late'], run)
        yp,yt=bkt_overall_predictions(data)
        m=all_metrics(yp,yt); aucs.append(m['auc']); eces.append(m['ece'])
        print(f"{name} run{run} | BKT AUC {m['auc']:.4f} ECE {m['ece']:.4f} | n {m['n']:,}")
    bkt_rows.append((name,np.mean(aucs),np.std(aucs,ddof=1),np.mean(eces),np.std(eces,ddof=1)))

print("\n"+"="*56)
print("BKT per-topic (mean±std over 5 splits)")
print(f"{'Topic':<12}{'BKT AUC':>17}{'BKT ECE':>17}")
print("-"*56)
for n,au,asd,ec,esd in bkt_rows:
    print(f"{n:<12}{au:>10.4f}±{asd:.4f}{ec:>10.4f}±{esd:.4f}")

import json
json.dump({n:{'auc_mean':au,'auc_std':asd,'ece_mean':ec,'ece_std':esd} for n,au,asd,ec,esd in bkt_rows},
          open(f"{OUTD}/bkt_per_topic.json","w"), indent=2)
print("\nsaved bkt_per_topic.json")

In [ ]:
# ============================================================================
# COLD START (Table 4): PC vs BKT, opportunity-stratified, pooled over topics, 5 splits
# Reuses the trained latent PC model and the same 5 splits as the main results.
# Stratifies test AUC by the per-skill opportunity index (how many times a skill has
# been seen before the current step), to compare PC and BKT in the low-data regime.
# ============================================================================

# --- PC eval with opportunity tracking ---
def evaluate_pc_with_opportunity(pnp, cluster_of, test, zero_tiers=False):
    K=len(cluster_of); C=int(cluster_of.max())+1
    latent = pnp.get('mode')=='latent' and pnp.get('tpl_diff') is not None and not zero_tiers
    dgo = np.zeros_like(pnp['dgo']) if (zero_tiers or latent) else pnp['dgo']
    dso = np.zeros_like(pnp['dso']) if (zero_tiers or latent) else pnp['dso']
    probs,labels,opp = [],[],[]
    for seq in test:
        seq=seq[:MAX_LEN]
        W=pnp['rp'].copy(); Q=[np.zeros((R,L)) for _ in range(C)]
        for c in range(C): Q[c][:,1]=pnp['cwh'][:,c]; Q[c][:,0]=1-pnp['cwh'][:,c]
        theta=[np.tile(pnp['leaf'][k],(R,1)) for k in range(K)]
        seen=np.zeros(K,dtype=int)
        for t,(k,x,pt,d,ti) in enumerate(seq):
            lk = pnp['learn'][k] if hasattr(pnp['learn'],'__len__') else pnp['learn']
            th=theta[k]; th=th*(1-pnp['forget'])+(1-th)*lk; theta[k]=th
            if latent:
                dd=pnp['tpl_diff'][ti] if ti<len(pnp['tpl_diff']) else 0.0
                g=sigmoid_np(pnp['sg'][k]+pnp['tgo'][pt]-dd); s=sigmoid_np(pnp['ss'][k]+pnp['tso'][pt]+dd)
            else:
                g=sigmoid_np(pnp['sg'][k]+pnp['tgo'][pt]+dgo[d]); s=sigmoid_np(pnp['ss'][k]+pnp['tso'][pt]+dso[d])
            c=cluster_of[k]; p_m=float((W*(Q[c]*th).sum(axis=1)).sum())
            if t>=1:
                probs.append(np.clip(p_m*(1-s)+(1-p_m)*g,1e-6,1-1e-6))
                labels.append(x); opp.append(int(seen[k]))
            seen[k]+=1
            pc=th*(1-s)+(1-th)*g; lik=pc if x==1 else 1-pc
            lik_r=(Q[c]*lik).sum(axis=1); nQ=Q[c]*lik
            Q[c]=nQ/np.clip(nQ.sum(axis=1,keepdims=True),1e-12,None)
            p1=(1-s) if x==1 else s; p0=g if x==1 else (1-g)
            num=th*p1; theta[k]=num/np.clip(num+(1-th)*p0,1e-12,None)
            ulw=np.log(np.clip(W,1e-12,None))+np.log(np.clip(lik_r,1e-12,None))
            W=np.exp(ulw-ulw.max()); W=W/W.sum()
    return np.array(probs), np.array(labels), np.array(opp,dtype=int)

# ---  BKT ---
from dataclasses import dataclass
from typing import Optional
EPS=1e-9
@dataclass
class BKTParams:
    p_init: float; p_learn: float; p_slip: float; p_guess: float
class BKTModel:
    def __init__(self, n_iter=100, tol=1e-5, init_params=None, random_state=0,
                 slip_bounds=(1e-4,0.3), guess_bounds=(1e-4,0.3), max_slip_plus_guess=0.9,
                 slip_prior_mean=0.1, slip_prior_strength=4.0,
                 guess_prior_mean=0.1, guess_prior_strength=4.0):
        self.n_iter=n_iter; self.tol=tol; self.random_state=random_state; self.init_params=init_params
        self.slip_bounds=slip_bounds; self.guess_bounds=guess_bounds
        assert 0<max_slip_plus_guess<1
        self.max_slip_plus_guess=max_slip_plus_guess
        self.slip_prior_mean=slip_prior_mean; self.slip_prior_strength=slip_prior_strength
        self.guess_prior_mean=guess_prior_mean; self.guess_prior_strength=guess_prior_strength
        self.params_=None
    @staticmethod
    def _pad(sequences):
        n=len(sequences); max_len=max((len(s) for s in sequences),default=0)
        obs=np.zeros((n,max_len),dtype=np.int8); mask=np.zeros((n,max_len),dtype=bool)
        for i,s in enumerate(sequences):
            Ls=len(s); obs[i,:Ls]=s; mask[i,:Ls]=True
        return obs,mask
    @staticmethod
    def _emission_prob(obs, slip, guess):
        p0=np.where(obs==1,guess,1-guess); p1=np.where(obs==1,1-slip,slip)
        return np.stack([p0,p1],axis=-1)
    def _e_step(self, obs, mask, params):
        N,T=obs.shape
        emis=np.clip(self._emission_prob(obs,params.p_slip,params.p_guess),EPS,1.0)
        trans=np.array([[1-params.p_learn,params.p_learn],[0.0,1.0]])
        prior=np.array([1-params.p_init,params.p_init])
        alpha=np.zeros((N,T,2)); c=np.ones((N,T))
        alpha0=prior[None,:]*emis[:,0,:]; c0=np.clip(alpha0.sum(1),EPS,None)
        alpha[:,0,:]=alpha0/c0[:,None]; c[:,0]=np.where(mask[:,0],c0,1.0)
        for t in range(1,T):
            pred=alpha[:,t-1,:]@trans; a_t=pred*emis[:,t,:]; c_t=np.clip(a_t.sum(1),EPS,None)
            ar=a_t/c_t[:,None]; m=mask[:,t]
            alpha[:,t,:]=np.where(m[:,None],ar,alpha[:,t-1,:]); c[:,t]=np.where(m,c_t,1.0)
        beta=np.ones((N,T,2))
        for t in range(T-2,-1,-1):
            nxt=beta[:,t+1,:]*emis[:,t+1,:]
            br=(nxt@trans.T)/np.clip(c[:,t+1],EPS,None)[:,None]; m=mask[:,t+1]
            beta[:,t,:]=np.where(m[:,None],br,beta[:,t+1,:])
        gamma=alpha*beta; gamma=gamma/np.clip(gamma.sum(-1,keepdims=True),EPS,None)
        xi=np.zeros((N,max(T-1,0),2,2))
        for t in range(T-1):
            for i in range(2):
                for j in range(2):
                    xi[:,t,i,j]=(alpha[:,t,i]*trans[i,j]*emis[:,t+1,j]*beta[:,t+1,j]/np.clip(c[:,t+1],EPS,None))
        gamma=gamma*mask[:,:,None]
        if T>1: xi=xi*mask[:,1:][:,:,None,None]
        ll=(np.log(np.clip(c,EPS,None))*mask).sum()
        return gamma,xi,ll
    def _m_step(self, obs, mask, gamma, xi):
        n=obs.shape[0]
        p_init=gamma[:,0,1].sum()/max(n,1)
        xs=xi.sum((0,1)); from0=xs[0,0]+xs[0,1]; p_learn=xs[0,1]/max(from0,EPS)
        mw=gamma[:,:,1]; nw=gamma[:,:,0]; inc=(obs==0); cor=(obs==1)
        ss=(mw*inc*mask).sum(); st=(mw*mask).sum(); gs=(nw*cor*mask).sum(); gt=(nw*mask).sum()
        p_slip=(ss+self.slip_prior_mean*self.slip_prior_strength)/max(st+self.slip_prior_strength,EPS)
        p_guess=(gs+self.guess_prior_mean*self.guess_prior_strength)/max(gt+self.guess_prior_strength,EPS)
        pc=lambda v: float(np.clip(v,1e-4,1-1e-4))
        p_init=pc(p_init); p_learn=pc(p_learn)
        p_slip=float(np.clip(p_slip,*self.slip_bounds)); p_guess=float(np.clip(p_guess,*self.guess_bounds))
        tot=p_slip+p_guess
        if tot>self.max_slip_plus_guess:
            sc=self.max_slip_plus_guess/tot; p_slip*=sc; p_guess*=sc
        return BKTParams(p_init,p_learn,p_slip,p_guess)
    def fit(self, sequences):
        sequences=[s for s in sequences if len(s)>0]
        if not sequences: raise ValueError("empty")
        obs,mask=self._pad(sequences)
        params=self.init_params or BKTParams(0.3,0.15,0.1,0.25)
        prev=-np.inf
        for it in range(self.n_iter):
            gamma,xi,ll=self._e_step(obs,mask,params); params=self._m_step(obs,mask,gamma,xi)
            if abs(ll-prev)<self.tol: break
            prev=ll
        self.params_=params; return self
    def predict_correctness_probabilities(self, sequences):
        obs,mask=self._pad(sequences); N,T=obs.shape; p=self.params_
        trans=np.array([[1-p.p_learn,p.p_learn],[0.0,1.0]]); prior=np.array([1-p.p_init,p.p_init])
        emis=np.clip(self._emission_prob(obs,p.p_slip,p.p_guess),EPS,1.0)
        belief=np.tile(prior,(N,1)); preds=np.zeros((N,T))
        for t in range(T):
            preds[:,t]=belief[:,0]*p.p_guess+belief[:,1]*(1-p.p_slip)
            upd=belief*emis[:,t,:]; upd=upd/np.clip(upd.sum(1,keepdims=True),EPS,None)
            belief=upd@trans
        return [preds[i,:int(mask[i].sum())] for i in range(N)]

# --- BKT predictions with opportunity index (returns: true, pred, opp) ---
def bkt_predictions_with_opportunity(data):
    K=data['n_skills']
    tr_by_skill={k:[] for k in range(K)}; te_by_skill={k:[] for k in range(K)}
    for seq in data['train']:
        per={}
        for (k,x,pt,d,ti) in seq: per.setdefault(k,[]).append(int(x))
        for k,arr in per.items(): tr_by_skill[k].append(np.array(arr,dtype=np.int8))
    for seq in data['test']:
        per={}
        for (k,x,pt,d,ti) in seq: per.setdefault(k,[]).append(int(x))
        for k,arr in per.items(): te_by_skill[k].append(np.array(arr,dtype=np.int8))
    yt,yp,opp=[],[],[]
    for k in range(K):
        if not tr_by_skill[k] or not te_by_skill[k]: continue
        try:
            m=BKTModel(n_iter=100).fit([s for s in tr_by_skill[k] if len(s)>0])
            preds=m.predict_correctness_probabilities(te_by_skill[k])
        except Exception: continue
        for true_arr,pred_arr in zip(te_by_skill[k],preds):
            yt.append(true_arr.astype(int)); yp.append(pred_arr); opp.append(np.arange(len(true_arr)))
    if not yt: return np.array([]),np.array([]),np.array([])
    return np.concatenate(yt),np.concatenate(yp),np.concatenate(opp)

# --- FULL RUN: all six topics, 5 runs, pooled ---
BINS=(0,1,2,3,5,10,10_000)
def _bl(lo,hi): return f"[{lo},{hi})" if hi<10_000 else f"{lo}+"
LABELS=[_bl(lo,hi) for lo,hi in zip(BINS[:-1],BINS[1:])]
def _bauc(pr,la,op):
    pr=np.asarray(pr,dtype=float); la=np.asarray(la,dtype=int); op=np.asarray(op,dtype=int)
    out={}
    for lo,hi in zip(BINS[:-1],BINS[1:]):
        m=(op>=lo)&(op<hi); lab=_bl(lo,hi)
        out[lab]=float(roc_auc_score(la[m],pr[m])) if (m.sum()>0 and len(np.unique(la[m]))>1) else float('nan')
    return out

pc_p,pc_l,pc_o=[],[],[]
bk_p,bk_l,bk_o=[],[],[]
for name in TOPICS_TO_RUN:
    sp=TOPIC_PREREQ_SPLIT[name]
    for run in range(RUNS):
        data=build_dataset_from_split(df, sp['early'], sp['late'], run)
        pnp_l=train_topic(data, seed=run, mode='latent')
        p,l,o=evaluate_pc_with_opportunity(pnp_l, data['cluster_of'], data['test'])
        pc_p.append(p); pc_l.append(l); pc_o.append(o)
        bl,bp,bo=bkt_predictions_with_opportunity(data)   # (true, pred, opp)
        bk_l.append(bl); bk_p.append(bp); bk_o.append(bo)
        print(f"{name} run{run} | PC n {len(p):,} | BKT n {len(bp):,}")

PCp=np.concatenate(pc_p); PCl=np.concatenate(pc_l); PCo=np.concatenate(pc_o)
BKp=np.concatenate(bk_p); BKl=np.concatenate(bk_l); BKo=np.concatenate(bk_o)
pcb=_bauc(PCp,PCl,PCo); bkb=_bauc(BKp,BKl,BKo)

print("\n"+"="*66)
print("TABLE 4 — cold start (pooled across six topics, 5 runs)")
print(f"{'Model':<12}"+"".join(f"{x:>9}" for x in LABELS))
print("-"*66)
print(f"{'BKT':<12}"+"".join(f"{bkb[x]:>9.4f}" for x in LABELS))
print(f"{'PC (latent)':<12}"+"".join(f"{pcb[x]:>9.4f}" for x in LABELS))
print(f"{'PC - BKT':<12}"+"".join(f"{pcb[x]-bkb[x]:>+9.4f}" for x in LABELS))

import json
json.dump({'BINS':list(BINS),'LABELS':LABELS,'BKT':bkb,'PC':pcb,
           'PC_minus_BKT':{x:pcb[x]-bkb[x] for x in LABELS}},
          open(f"{OUTD}/cold_start_table4.json","w"), indent=2)
print("\nsaved cold_start_table4.json")

In [ ]:
# ==== DKT baseline: single-layer LSTM knowledge tracing ====
# Standard Deep Knowledge Tracing (Piech et al. 2015). Input at each step is a one-hot
# encoding of (skill, correctness) of width 2*n_skills; a single-layer LSTM (hidden=100)
# outputs a per-skill correctness probability. At step t we read off the probability for
# the skill asked at t+1 (predict-before-update, so no leakage), scoring steps t>=1.
# Trained with Adam + BCE on masked next-step targets, gradient clipping, and early
# stopping on a held-out validation AUC (best-AUC checkpoint restored).

class DKT(nn.Module):
    def __init__(self, n_skills, hidden=DKT_HIDDEN, dropout=DKT_DROPOUT):
        super().__init__()
        self.lstm=nn.LSTM(2*n_skills, hidden, batch_first=True)
        self.drop=nn.Dropout(dropout); self.out=nn.Linear(hidden, n_skills)
    def forward(self,x):
        h,_=self.lstm(x); return self.out(self.drop(h))

def dkt_tensors(seqs, K):
    B=len(seqs); T=min(MAX_LEN,max(len(s) for s in seqs))
    X=np.zeros((B,T,2*K),np.float32); SK=np.zeros((B,T),np.int64)
    Y=np.zeros((B,T),np.float32); M=np.zeros((B,T),np.float32)
    for b,seq in enumerate(seqs):
        for t,step in enumerate(seq[:T]):
            k,c=step[0],step[1]
            X[b,t,k+c*K]=1.0; SK[b,t]=k; Y[b,t]=c
            if t>=1: M[b,t]=1.0
    tt=lambda a,d: torch.tensor(a,dtype=d,device=device)
    return tt(X,torch.float32),tt(SK,torch.long),tt(Y,torch.float32),tt(M,torch.float32)

def _dkt_predict(model, seqs, K, bs=256):
    model.eval(); P,Lb=[],[]
    with torch.no_grad():
        for i in range(0,len(seqs),bs):
            X,SK,Y,M=dkt_tensors(seqs[i:i+bs],K)
            pred=torch.sigmoid(model(X)[:,:-1,:])
            ts,ty,ms=SK[:,1:],Y[:,1:],M[:,1:]
            pk=torch.gather(pred,2,ts.unsqueeze(2)).squeeze(2); sel=ms>0
            P.append(pk[sel].cpu().numpy()); Lb.append(ty[sel].cpu().numpy())
    return np.concatenate(P), np.concatenate(Lb)

def train_eval_dkt(data, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    K=data['n_skills']; model=DKT(K).to(device)
    n_params=sum(p.numel() for p in model.parameters())
    opt=torch.optim.Adam(model.parameters(),lr=DKT_LR); bce=nn.BCEWithLogitsLoss(reduction='none')
    tr=list(data['train']); rng=np.random.default_rng(seed); rng.shuffle(tr)
    n_val=max(1,int(len(tr)*DKT_VAL_FRAC)); val,tr = tr[:n_val], tr[n_val:]
    best_auc,best_state=-1.0,None
    for ep in range(DKT_EPOCHS):
        model.train(); rng.shuffle(tr)
        for i in range(0,len(tr),DKT_BATCH):
            X,SK,Y,M=dkt_tensors(tr[i:i+DKT_BATCH],K)
            logits=model(X)[:,:-1,:]
            ts,ty,ms=SK[:,1:],Y[:,1:],M[:,1:]
            pk=torch.gather(logits,2,ts.unsqueeze(2)).squeeze(2)
            loss=(bce(pk,ty)*ms).sum()/ms.sum().clamp(min=1)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        vp,vl=_dkt_predict(model,val,K)
        vauc=roc_auc_score(vl,vp) if len(np.unique(vl))>1 else 0.5
        if vauc>best_auc:
            best_auc=vauc; best_state={k:v.detach().clone() for k,v in model.state_dict().items()}
    if best_state is not None: model.load_state_dict(best_state)
    p,l=_dkt_predict(model,data['test'],K)
    return p,l,n_params,float(best_auc)

In [ ]:

# ====  main experiment loop ====
# For each of the six topics and five splits, trains four model variants on the same
# training data and evaluates all four on the same held-out test set:
#   base  — no latent difficulty (tier-based emission only)
#   obs   — same base model, evaluated with tier offsets zeroed out (observed-difficulty ablation)
#   lat   — full model: latent per-template difficulty + learned forgetting (the reported PC result)
#   noF   — latent difficulty with forgetting fixed at f=0 (forgetting ablation)
# DKT is trained once per (topic, run) on the same split for comparison. An assertion
# checks all four PC variants and DKT score the identical held-out set, so AUCs are
# directly comparable. Results are checkpointed per (topic, run) to res_<topic>_run<run>.json
# so a killed Kaggle session can resume without recomputation.

for name in TOPICS_TO_RUN:
    sp=TOPIC_PREREQ_SPLIT[name]
    for run in range(RUNS):
        respath=f"{RESDIR}/res_{name}_run{run}.json"
        if os.path.exists(respath):
            print(f"skip {name} run{run} (already done)"); continue
        t0=timer.time()
        data=build_dataset_from_split(df, sp['early'], sp['late'], run)

        pnp_o = train_topic(data, seed=run, mode='tiers')
        bp,bl = evaluate_pc(pnp_o, data['cluster_of'], data['test'], zero_tiers=True)
        op,ol = evaluate_pc(pnp_o, data['cluster_of'], data['test'], zero_tiers=False)
        pnp_l = train_topic(data, seed=run, mode='latent')
        lp,ll = evaluate_pc(pnp_l, data['cluster_of'], data['test'])
        pnp_nf = train_topic_best(data, seed=run, restarts=1, mode='latent', no_forget=True)
        np_,nl = evaluate_pc(pnp_nf, data['cluster_of'], data['test'])
        dp,dl,d_par,d_val = train_eval_dkt(data, seed=run)

        assert len(lp)==len(dp)==len(np_)==len(bp), \
            f"EVAL SET MISMATCH: base {len(bp)} / lat {len(lp)} / noF {len(np_)} / DKT {len(dp)}"

        rec = dict(topic=name, run=run, device=str(device), secs=round(timer.time()-t0,1),
                   n_eval=int(len(lp)), dkt_val_auc=d_val,
                   pc_params=pnp_l['n_params'], dkt_params=int(d_par),
                   forget_learned=pnp_l['forget'], forget_ablated=pnp_nf['forget'],
                   base={f"base_{k}":v for k,v in all_metrics(bp,bl).items()},
                   obs ={f"obs_{k}":v  for k,v in all_metrics(op,ol).items()},
                   lat ={f"lat_{k}":v  for k,v in all_metrics(lp,ll).items()},
                   nof ={f"nof_{k}":v  for k,v in all_metrics(np_,nl).items()},
                   dkt ={f"dkt_{k}":v  for k,v in all_metrics(dp,dl).items()})
        flat = {**{k:v for k,v in rec.items() if not isinstance(v,dict)},
                **rec['base'],**rec['obs'],**rec['lat'],**rec['nof'],**rec['dkt']}
        json.dump(flat, open(respath,"w"), indent=2)
        print(f"{name} run{run} | AUC base {flat['base_auc']:.4f} obs {flat['obs_auc']:.4f} "
              f"lat {flat['lat_auc']:.4f} noF {flat['nof_auc']:.4f} DKT {flat['dkt_auc']:.4f} "
              f"| ECE base {flat['base_ece']:.4f} lat {flat['lat_ece']:.4f} "
              f"noF {flat['nof_ece']:.4f} DKT {flat['dkt_ece']:.4f} "
              f"| f_learn {flat['forget_learned']:.4f} | n {flat['n_eval']:,} | {flat['secs']}s")

In [ ]:
# ==== CELL : Aggregate -> tables ====
# Loads every per-(topic, run) result JSON written by the main experiment loop, saves the
# combined table as all_runs.csv, and prints the summary tables (mean±std over 5 splits):
#   Table 1 — PC (base) vs PC + latent vs DKT, and the gap (PC-DKT)
#   Table 2 — forgetting ablation: exact f=0 vs learned f, AUC and ECE
# These are the source numbers for the paper's main comparison and ablation tables.

runs_df = pd.DataFrame([json.load(open(f)) for f in sorted(glob.glob(f"{RESDIR}/res_*.json"))])
runs_df.to_csv(f"{OUTD}/all_runs.csv", index=False)
print(f"\nloaded {len(runs_df)} runs:\n{runs_df['topic'].value_counts().to_string()}\n")

ORDER=[t for t in TOPICS_TO_RUN if t in set(runs_df['topic'])]
def ms(g,c): return g[c].mean(), g[c].std(ddof=1)

print("="*96)
print("TABLE 1 — main results (AUC, mean±std)")
print(f"{'Topic':<12}{'PC (base)':>18}{'PC + latent':>18}{'DKT':>18}{'gap (PC-DKT)':>20}")
print("-"*96)
for t in ORDER:
    g=runs_df[runs_df.topic==t]; gap=g['lat_auc']-g['dkt_auc']
    print(f"{t:<12}{ms(g,'base_auc')[0]:>11.4f}±{ms(g,'base_auc')[1]:.4f}"
          f"{ms(g,'lat_auc')[0]:>11.4f}±{ms(g,'lat_auc')[1]:.4f}"
          f"{ms(g,'dkt_auc')[0]:>11.4f}±{ms(g,'dkt_auc')[1]:.4f}"
          f"{gap.mean():>+13.4f}±{gap.std(ddof=1):.4f}")

print("\n"+"="*96)
print("TABLE 2 — forgetting ablation (exact f=0 vs learned f)")
print(f"{'Topic':<12}{'no-forget AUC':>18}{'forget AUC':>18}{'no-forget ECE':>18}{'forget ECE':>18}")
print("-"*96)
for t in ORDER:
    g=runs_df[runs_df.topic==t]
    print(f"{t:<12}{ms(g,'nof_auc')[0]:>11.4f}±{ms(g,'nof_auc')[1]:.4f}"
          f"{ms(g,'lat_auc')[0]:>11.4f}±{ms(g,'lat_auc')[1]:.4f}"
          f"{ms(g,'nof_ece')[0]:>11.4f}±{ms(g,'nof_ece')[1]:.4f}"
          f"{ms(g,'lat_ece')[0]:>11.4f}±{ms(g,'lat_ece')[1]:.4f}")

print("\n"+"="*96)
print("TABLE 3 — calibration ECE (10 bins, fixed-width)")
print(f"{'Topic':<12}{'DKT':>16}{'PC (base)':>16}{'PC + observed':>16}{'PC + latent':>16}")
print("-"*96)
for t in ORDER:
    g=runs_df[runs_df.topic==t]
    print(f"{t:<12}{ms(g,'dkt_ece')[0]:>10.4f}±{ms(g,'dkt_ece')[1]:.4f}"
          f"{ms(g,'base_ece')[0]:>10.4f}±{ms(g,'base_ece')[1]:.4f}"
          f"{ms(g,'obs_ece')[0]:>10.4f}±{ms(g,'obs_ece')[1]:.4f}"
          f"{ms(g,'lat_ece')[0]:>10.4f}±{ms(g,'lat_ece')[1]:.4f}")
print("="*96)

print("\nPaired per-run deltas (positive = first term better):")
for t in ORDER:
    g=runs_df[runs_df.topic==t]
    d1=g['lat_auc']-g['dkt_auc']; d2=g['lat_auc']-g['nof_auc']; d3=g['nof_ece']-g['lat_ece']
    print(f"  {t:<12} PC-DKT AUC {d1.mean():+.4f} (min {d1.min():+.4f}) | "
          f"forget gain AUC {d2.mean():+.4f} (min {d2.min():+.4f}) | "
          f"forget gain ECE {d3.mean():+.4f} (min {d3.min():+.4f})")


In [ ]:
# ==== CELL : paired significance tests + experiment config ====
# Paired tests over the 5 fixed splits (same splits for every model, so a paired test is
# valid): PC+latent vs DKT (t-test and Wilcoxon), and PC+latent vs the no-forget ablation
# (t-test). With n=5 splits, the Wilcoxon signed-rank test's smallest attainable p-value
# is 0.0625, so it can never reach the conventional 0.05 threshold here — reported for
# completeness alongside the paired t-test, not as the primary significance claim.
# Also dumps experiment_config.json: all hyperparameters, the dataset's MD5 hash (for data
# provenance), and library/GPU versions, so the run is fully specified for reproduction.
print("\n"+"="*96)
print("SIGNIFICANCE (paired over the 5 fixed splits)")
print(f"{'Topic':<12}{'PC-DKT gap':>14}{'t p-value':>12}{'wilcoxon p':>13}   "
      f"{'forget gain':>13}{'t p-value':>12}")
print("-"*96)
for t in ORDER:
    g=runs_df[runs_df.topic==t]
    gap=g['lat_auc']-g['dkt_auc']; fg=g['lat_auc']-g['nof_auc']
    _,p_t=ttest_rel(g['lat_auc'],g['dkt_auc'])
    try:    _,p_w=wilcoxon(g['lat_auc'],g['dkt_auc'])
    except Exception: p_w=float('nan')
    _,p_f=ttest_rel(g['lat_auc'],g['nof_auc'])
    print(f"{t:<12}{gap.mean():>+14.4f}{p_t:>12.5f}{p_w:>13.4f}   {fg.mean():>+13.4f}{p_f:>12.5f}")
print("Note: with n=5 the smallest attainable Wilcoxon p-value is 0.0625.")
print("="*96)

json.dump(dict(runs=RUNS, topics=TOPICS_TO_RUN, max_len=MAX_LEN, min_interact=MIN_INTERACT,
               test_size=TEST_SIZE, n_tiers=N_TIERS, ref_tier=REF_TIER, R=R, L=L,
               pc_epochs=PC_EPOCHS, pc_lr=PC_LR, pc_shrink=PC_SHRINK, pc_clip=PC_CLIP,
               dkt_hidden=DKT_HIDDEN, dkt_epochs=DKT_EPOCHS, dkt_lr=DKT_LR,
               dkt_batch=DKT_BATCH, dkt_val_frac=DKT_VAL_FRAC,
               use_learnable_leaf=USE_LEARNABLE_LEAF,
               ablation="exact f=0 (decay term removed from transition)",
               data_file=DATA_FILE, data_md5=hashlib.md5(open(DATA_FILE,'rb').read()).hexdigest(),
               n_rows=int(df.shape[0]), n_students=int(df['user_id'].nunique()),
               n_skills=int(df['skill'].nunique()),
               torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__,
               device=str(device),
               gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)),
          open(f"{OUTD}/experiment_config.json","w"), indent=2)
print("\nSaved: results/res_*.json, all_runs.csv, experiment_config.json")

In [ ]:
# ==== Per-topic dataset statistics ====
# For each topic, reports the total interactions and unique students for that topic's
# skills (across the full dataset), plus the train/test sequence counts from split 0
# (after the <3-interaction filter and the 80/20 student-level split). Used for the
# per-topic dataset-statistics table in the appendix.

print(f"{'Topic':<12}{'Interactions':>14}{'Students':>10}{'Train seq':>10}{'Test seq':>10}")
print("-"*56)
for name in TOPICS_TO_RUN:
    sp = TOPIC_PREREQ_SPLIT[name]
    # all rows for this topic's skills
    order = [s for s in sp['early']+sp['late'] if s in df['skill'].unique()]
    sub = df[df['skill'].isin(order)]
    n_inter = len(sub)
    n_stud = sub['user_id'].nunique()
    # train/test sequence counts from split 0
    data = build_dataset_from_split(df, sp['early'], sp['late'], 0)
    n_tr = len(data['train']); n_te = len(data['test'])
    print(f"{name:<12}{n_inter:>14,}{n_stud:>10,}{n_tr:>10,}{n_te:>10,}")